In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window

BRONZE_VOLUME = "/Volumes/ev_catalog/bronze/api/charge_cards/"
SILVER_TABLE = "ev_catalog.silver.sl_charge_cards"
SILVER_PATH = "abfss://silver@evdatalakedevnich.dfs.core.windows.net/api/charge_cards/"
QUARANTINE_PATH = "abfss://silver@evdatalakedevnich.dfs.core.windows.net/quarantine/charge_cards/"

dbutils.widgets.text("p_run_date", "", "Run date YYYY-MM-DD (blank=full load)")
dbutils.widgets.text("p_run_type", "", "Run type full/incremental (blank=full load)")
run_date = dbutils.widgets.get("p_run_date").strip()
run_type = dbutils.widgets.get("p_run_type").strip()

In [0]:
from pyspark.sql.types import StructType, StructField, ArrayType, StringType, LongType, DoubleType, BooleanType
if run_date and run_type == "incremental":
    bronze_path = f"{BRONZE_VOLUME}ingestion_date={run_date}"
else:
    bronze_path = BRONZE_VOLUME



CHARGE_CARD_SCHEMA = StructType([
    StructField("data", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("card_id", StringType(), True),
        StructField("card_number", StringType(), True),
        StructField("card_type", StringType(), True),
        StructField("created_at", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("rfid_uid", StringType(), True),
        StructField("status", StringType(), True),
        StructField("updated_at", StringType(), True)
    ])), True),
                StructField("pagination", StructType([
                    StructField("page", LongType()),
                    StructField("page_size", LongType()),
                    StructField("total_pages", LongType()),
                    StructField("total_records", LongType(), True),
                    ]), True),
                ])

df_charges = spark.read.format("json")\
                .schema(CHARGE_CARD_SCHEMA)\
                .option("multiline", "true")\
                .option("recursiveFileLookup", "true")\
                .load(bronze_path)\
                .withColumn("_src", F.col("_metadata.file_path"))\
                .filter(F.col("data").isNotNull() & (F.size(F.col("data")) > 0))

print(f"Pages with data: {df_charges.count()}")


In [0]:
df_exploded = df_charges.withColumn("record", F.explode(F.col("data")))\
                        .select(F.col("record.id").cast("long").alias("id"),
                                F.col("record.card_id").alias("card_id"),
                                F.col("record.card_number").alias("card_number"),
                                F.col("record.card_type").alias("card_type"),
                                F.col("record.created_at").cast("timestamp").alias("created_at"),
                                F.col("record.customer_id").alias("customer_id"),
                                F.col("record.rfid_uid").alias("rfid_uid"),
                                F.col("record.status").alias("status"),
                                F.col("record.updated_at").cast("timestamp").alias("updated_at"))

total_rows = df_exploded.count()
print(f"Total rows: {total_rows:,}")

if total_rows == 0:
        print("Total rows are zero. Stopping execution")
        dbutils.notebook.exit("NO_DATA")
df_exploded.show(3, truncate=80)

In [0]:
# Filter bad records
df_checked = df_exploded.withColumn("rejection_reason",
                                     F.when(F.col("card_id").isNull(), "null_card_id")
                                     .when(F.col("customer_id").isNull(), "null_customer_id")
                                     .when(F.col("card_number").isNull(), "null_card_number")
                                     .when(F.col("status").isNull(), "null_status")
                                     .otherwise(None))

df_bad = df_checked.filter(F.col("rejection_reason").isNotNull())
# Filter good records
df_good = df_checked.filter(F.col("rejection_reason").isNull()).drop("rejection_reason")

bad_count = df_bad.count()
if bad_count > 0:
    print(f"Found {bad_count:,} bad records")
    df_bad.withColumn("quarantine_ts", F.current_timestamp).write.format("delta").mode("append").saveAsTable(QUARANTINE_PATH)
else:
    print("No bad records - quarantine not written")

In [0]:
# deduplication
w = Window.partitionBy("card_id").orderBy(F.col("updated_at").desc())
df_dedup = df_good.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn", "_src")

In [0]:
# Timezone conversion UTC to AEST
df_tz = (
    df_dedup.withColumn("created_at_utc", F.to_timestamp("created_at"))
                .withColumn("updated_at_utc", F.to_timestamp("updated_at"))
                .withColumn("created_at_aest", F.from_utc_timestamp("created_at_utc", "Australia/Sydney"))
                .withColumn("updated_at_aest", F.from_utc_timestamp("updated_at_utc", "Australia/Sydney"))
                .drop("created_at", "updated_at")
)
        

In [0]:
# PCI masking (card_number --> last 4 digits)
df_pci = (
    df_tz.withColumn("card_last4", F.regexp_replace(F.col("card_number"),  r"^.*(\d{4})$", r"$1"))
          .drop("card_number")
)


In [0]:
# silver metadata columns
df_silver = (
    df_pci.withColumn("sl_ingested_at", F.current_timestamp())
          .withColumn("sl_batch_id", F.lit(run_date if run_date else "full"))
)

In [0]:
df_silver.show(3, truncate=80)

In [0]:
# ── Cell 10: CREATE TABLE IF NOT EXISTS ───────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
    id               LONG,
    card_id          STRING,
    customer_id      STRING,
    rfid_uid         STRING,
    card_last4       STRING,
    card_type        STRING,
    status           STRING,
    created_at_utc   TIMESTAMP,
    created_at_aest  TIMESTAMP,
    updated_at_utc   TIMESTAMP,
    updated_at_aest  TIMESTAMP,
    sl_ingested_at   TIMESTAMP,
    sl_batch_id      STRING
)
USING DELTA
LOCATION '{SILVER_PATH}'
""")

In [0]:
# ── Cell 11: MERGE (upsert on card_id) ───────────────────────────────────────
from delta.tables import DeltaTable
tgt = DeltaTable.forName(spark, SILVER_TABLE)

merge_result = (
    tgt.alias("t")
       .merge(df_silver.alias("s"), "t.card_id = s.card_id")
       .whenMatchedUpdateAll()
       .whenNotMatchedInsertAll()
       .execute()
)

print(f"MERGE done: {SILVER_TABLE}")

In [0]:
# ── Cell 12: run stats ────────────────────────────────────────────────────────
merge_metrics  = merge_result.collect()[0]
rows_inserted  = int(merge_metrics["num_inserted_rows"])
rows_updated   = int(merge_metrics["num_updated_rows"])
rows_in_source = int(df_silver.count())
rows_duplicate = rows_in_source - rows_inserted - rows_updated

print("\n" + "=" * 55)
print(f"  PIPELINE RUN STATS   batch_id = {run_date if run_date else 'full'}")
print("=" * 55)
print(f"  Source records read (after explode + dedup) : {rows_in_source:>10,}")
print(f"  Quarantined (bad quality)                   : {bad_count:>10,}")
print(f"  NEW rows inserted into Silver               : {rows_inserted:>10,}")
print(f"  EXISTING rows updated in Silver             : {rows_updated:>10,}")
print(f"  Duplicates removed before MERGE             : {rows_duplicate:>10,}")
print("=" * 55)

if rows_inserted == 0 and rows_updated == 0:
    print("  WARNING: Zero rows written — check Bronze partition exists for this date.")
elif rows_inserted == 0 and rows_updated > 0:
    print("  INFO: All records already existed — pipeline re-run, Silver is up to date.")
else:
    print("  OK: Silver updated successfully.")
print("=" * 55)

In [0]:
# ── Cell 13: post-load breakdown ──────────────────────────────────────────────
spark.sql(f"""
SELECT
    card_type,
    status,
    COUNT(*)   AS record_count,
    COUNT(CASE WHEN card_last4 IS NOT NULL THEN 1 END) AS masked_cards
FROM {SILVER_TABLE}
GROUP BY card_type, status
ORDER BY record_count DESC
""").display()

In [0]:
SILVER_PATH

In [0]:
SILVER_TABLE